# Self-Editing Coding Agent with LangGraph

This notebook builds an agent that can **read, edit, and run Python files**. Every LangGraph and LangChain building block is explained right before the cell that uses it.

**The final graph looks like this:**

```
START -> agent --(tool call?)--> tools -> agent -> ... -> END
                 \--(no tool call)--> END
```

## 1. Install and import

- **`langgraph`**: the graph framework (state, nodes, edges, prebuilt helpers).
- **`langchain`** and **`langchain-groq`**: model loading, tools, and message types.
- **`python-dotenv`**: loads `GROQ_API_KEY` from a `.env` file.

In [ ]:
%pip install -q langgraph langchain langchain-groq python-dotenv

### Imports explained

| Import | What it is |
|---|---|
| `StateGraph` | The graph builder. You give it a state schema, then add nodes and edges. |
| `START` | A special built-in node name meaning "entry point of the graph". |
| `END` | A special built-in node name meaning "stop the graph here". |
| `add_messages` | A **reducer**. It tells LangGraph how to merge new messages into the state (append, don't overwrite). |
| `ToolNode` | A prebuilt node that executes whatever tool calls the model requested. |
| `tools_condition` | A prebuilt routing function that returns `"tools"` or `END`. |
| `init_chat_model` | LangChain helper that creates a chat model from a name and provider. |
| `@tool` | Decorator that turns a Python function into a tool the model can call. |
| `SystemMessage`, `HumanMessage` | Message types for instructions and user input. |

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("API Key not found")

## 2. Define the tools

### `@tool`
The decorator reads the function's **name**, **type hints**, and **docstring** and turns them into a schema the model sees. This is why:
- the docstring matters, because the model uses it to decide *when* to call the tool;
- type hints matter, because they define the arguments the model must supply.

The three tools below are the agent's abilities: read a file, write a file, run a script.

In [ ]:
@tool
def read_file(file_input: str) -> str:
    """Read and return the contents of an existing text file."""
    file_path = Path(file_input)
    if not file_path.is_file():
        return f"Error: '{file_input}' is not a file."
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(file_input: str, content: str) -> str:
    """Write modified code to an existing file."""
    file_path = Path(file_input)
    file_path.write_text(content, encoding="utf-8")
    return "File written successfully"


@tool
def run_code(script_path: str) -> str:
    """Execute a Python script and return its output."""
    command = [sys.executable, script_path]
    if Path(script_path).name == "lg_prac.py":
        command.append("--test")
    try:
        result = subprocess.run(
            command, capture_output=True, text=True, timeout=20
        )
    except subprocess.TimeoutExpired:
        return "Error: script timed out after 20 seconds"
    return (
        f"Exit code: {result.returncode}\n"
        f"Output: {result.stdout}\n"
        f"Errors: {result.stderr}"
    )

Tools return **strings** (including error strings) instead of raising exceptions. That way the model *sees* the failure as a normal result and can react to it, for example by fixing a wrong path.

## 3. System prompt

This is the instruction set the model receives on every call. It is **not** stored in the graph state; we prepend it inside the agent node (step 5).

In [ ]:
SYSTEM_PROMPT = """
You are a self-editing Python coding agent.

Project files:
- lg_tools.py: Contains tools and model configuration.

Rules:
1. Always inspect the existing code before editing.
2. When modifying tools, read lg_tools.py.
3. When modifying the graph, read lg_prac.py.
4. Never pass "." or a directory to read_file.
5. Always provide the exact file path.
6. If the file path is unknown, ask the user.
7. Edit only the requested file.
8. Run the modified code and check the output.
"""

## 4. State, tools list, and model

### State (`TypedDict` + `add_messages`)
The **state** is the shared data every node reads and updates. Ours has one key, `messages`.

`Annotated[list, add_messages]` attaches a **reducer** to that key:
- without it, a node returning `{"messages": [new]}` would **overwrite** the list;
- with `add_messages`, the new message is **appended** (and messages with the same ID are updated).

### `init_chat_model`
Creates a chat model from a name and a provider, so you can swap providers without changing the rest of the code.

### `bind_tools`
Attaches the tool schemas to the model. Now when the model answers, it can either reply with text **or** return a `tool_calls` list such as `[{"name": "read_file", "args": {...}}]`. Binding does **not** run the tools; it only lets the model request them.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


tools = [read_file, write_file, run_code]

model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    api_key=api_key,
)
model_with_tools = model.bind_tools(tools)

## 5. The agent node

A **node** is just a function: it takes the current state and returns a **partial state update**.

- `SystemMessage(...)` is prepended on every call, so the instructions are always fresh and are never duplicated in the saved history.
- `model_with_tools.invoke(...)` sends the messages to the model. The reply is either text or a tool request.
- Returning `{"messages": [response]}` goes through `add_messages`, so the reply is appended to the history.

In [ ]:
def agent_node(state: AgentState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

## 6. Build the graph

### `StateGraph(AgentState)`
Creates an empty graph builder bound to our state schema.

### `add_node(name, fn)`
Registers a node. The name is how edges refer to it.
- `"agent"`: our function that calls the model.
- `"tools"`: a **`ToolNode`**, a prebuilt node that looks at the last message's `tool_calls`, runs each matching tool, and appends the results as `ToolMessage`s. You don't write any execution code.

### `add_edge(a, b)`
A **fixed** edge: after `a` finishes, always go to `b`.
- `START -> agent`: begin by calling the model.
- `tools -> agent`: after tools run, return to the model so it can see the results. This loop is what makes it an agent: *think, act, observe, repeat*.

### `add_conditional_edges(source, router)`
A **dynamic** edge: after `source` finishes, the `router` function decides where to go next. `tools_condition` is the prebuilt router:
- last message **has** tool calls -> returns `"tools"`
- last message **has no** tool calls -> returns `END`

That second case is where `END` comes in, so you don't need to wire it yourself.

### `compile()`
Validates the graph (no dangling nodes, edges point to real nodes) and returns a runnable object. You must compile before you can call `invoke` or `stream`.

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # -> "tools" or END
builder.add_edge("tools", "agent")                       # loop back after tool use

graph = builder.compile()

### Optional: the explicit version of `tools_condition`
This does exactly what `tools_condition` does, with `END` visible. Not needed to run the notebook; it's here for understanding.

```python
def route(state: AgentState):
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
```
The third argument is a **path map**: it translates the router's return values into node names.

## 7. Inspect the graph

`graph.get_graph()` returns a drawable description of the compiled graph. `draw_mermaid()` outputs Mermaid text you can paste into any Mermaid viewer. It needs no network access. (`draw_mermaid_png()` renders an image but may call an external service.)

In [ ]:
print(graph.get_graph().draw_mermaid())

## 8. Run the agent

### `invoke(input, config)`
Runs the graph from `START` to `END` and returns the **final state**.
- **Input**: a partial state, here `{"messages": [HumanMessage(...)]}`.
- **Output**: the full final state, so `result["messages"][-1]` is the agent's last reply.
- **`recursion_limit`** (in `config`): the maximum number of steps (node executions) before LangGraph raises an error. It guards against infinite agent loops. Each agent or tools hop counts as a step.

In [ ]:
def ask(question: str):
    result = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        {"recursion_limit": 25},
    )
    return result["messages"][-1].content

# Example (change the file path to a real file):
# print(ask("Read lg_tools.py and tell me what it does"))

### `stream(input, config, stream_mode=...)`
Same as `invoke`, but yields updates **as the graph runs**, so you can watch each step happen.
- `stream_mode="updates"`: yields `{node_name: state_update}` after each node finishes.
- `stream_mode="values"`: yields the full state after each step.

This is very useful for debugging, since you can see every tool call and result.

In [ ]:
def ask_streaming(question: str):
    for step in graph.stream(
        {"messages": [HumanMessage(content=question)]},
        {"recursion_limit": 25},
        stream_mode="updates",
    ):
        for node_name, update in step.items():
            msg = update["messages"][-1]
            print(f"--- {node_name} ---")
            if getattr(msg, "tool_calls", None):
                print("Tool calls:", msg.tool_calls)
            else:
                print(msg.content)

# Example:
# ask_streaming("Read lg_tools.py and summarize it")

## 9. Interactive chat loop

A simple loop for talking to the agent. Type `exit` to stop.

In a notebook, `input()` shows a text box under the cell. Note that the `--test` handling from the script version isn't needed here: notebooks don't re-run themselves as a subprocess unless your agent calls `run_code` on this file.

In [ ]:
print("Agent ready. Type 'exit' to quit.")
while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break
    print("\nAgent:", ask(user_input))

## Quick reference

| Concept | Role in this graph |
|---|---|
| `AgentState` + `add_messages` | Shared memory; new messages are appended |
| `StateGraph` | Builder for the graph |
| `add_node` | Registers `agent` and `tools` |
| `START` / `END` | Entry and exit points |
| `add_edge` | Fixed transitions (`START->agent`, `tools->agent`) |
| `add_conditional_edges` + `tools_condition` | Chooses `tools` or `END` after the agent |
| `ToolNode` | Executes the tool calls the model requested |
| `compile()` | Finalizes the graph into something runnable |
| `invoke` / `stream` | Run it, either all at once or step by step |
| `recursion_limit` | Safety cap on loop iterations |